# Wavebit Tutorial 3
Device - CPU and GPU

## Running the GPU sections

The GPU cells below need a **CUDA** device. They are guarded: without one they print a
notice and compute nothing. That is deliberate — CPU timings printed under a "GPU"
heading would be read as GPU timings, and a misleading number is worse than no number.

### Apple Silicon (Metal / MPS) will not work here

This is not a configuration you can switch on. Wavebit solves in **complex128**
end to end, and Metal supports neither `float64` nor `complex128`:

```
torch.zeros(2, dtype=torch.float32,    device='mps')   # OK
torch.zeros(2, dtype=torch.complex64,  device='mps')   # OK
torch.zeros(2, dtype=torch.float64,    device='mps')   # RuntimeError
torch.zeros(2, dtype=torch.complex128, device='mps')   # RuntimeError
```

That rules out both the PyTorch MPS backend and Apple's experimental `jax-metal`
plugin, which has the same limitation.

Dropping to single precision is not a setting either. The solve is an eigendecomposition
of large non-symmetric complex matrices; in float32 it loses roughly eight significant
digits, which is enough to break agreement between the numpy, jax and torch backends.
Supporting Metal would mean designing a separate single-precision path and deciding what
accuracy it is allowed to claim — a project, not a flag.

So: to fill in the GPU cells, run this notebook on a CUDA machine.

In [1]:
import warnings
# The 32-bit runs below trip UnresolvedBranchWarning on purpose -- it is the cost this
# tutorial is about. Python's default format prefixes the message with the absolute
# path of the source line, which must not ship in a stored output (docs/README.md, and
# tests/test_docs_contract.py enforces it), so print the category and message only.
warnings.formatwarning = lambda msg, cat, *a, **k: f'{cat.__name__}: {msg}\n'

import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = '0'

import time
import numpy as np

import wavebit

import torch

# The GPU cells below are skipped unless this is True. Apple Metal (MPS) does not
# count: it has no float64/complex128, which this solver needs throughout.
GPU_AVAILABLE = torch.cuda.is_available()
print('CUDA GPU available:', GPU_AVAILABLE)


CUDA GPU available: False


In [2]:
# experiment options
psi = np.pi / 2  # pure s (TE-like); psi is the linear angle

n_top = 1  # n_topncidence
n_bot = 1  # n_transmission

theta = 20 * np.pi / 180
phi = 50 * np.pi / 180

wavelength = 900

thickness = [500]
period = [1000, 1000]

fto = [15, 15]
res_x, res_y, res_z = 20, 20, 20

ucell = np.array([
            [
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ],
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ],
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ],
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ],
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ],
                [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, ],
                [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, ],
                [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, ],
                [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, ],
                [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, ],
            ],
        ]) * 4 + 1

## set device

1. at initialization

In [3]:
backend = 2  # PyTorch backend
device = 0 # CPU;
dtype = 0
solver = wavebit.rcwa(backend=backend, psi=psi, n_top=n_top, n_bot=n_bot, theta=theta, phi=phi,
                     fto=fto, wavelength=wavelength, period=period, ucell=ucell,
                     thickness=thickness, device=device, type_complex=dtype)

    

2. after initialization

In [4]:
backend = 2  # PyTorch backend

solver = wavebit.rcwa(backend=backend, psi=psi, n_top=n_top, n_bot=n_bot, theta=theta, phi=phi,
                     fto=fto, wavelength=wavelength, period=period, ucell=ucell,
                     thickness=thickness, type_complex=dtype)
solver.device = 0

# Test

## PyTorch

### CPU, 64 bit

In [5]:
backend = 2  # PyTorch backend
device = 0 # CPU;
dtype = 0
solver = wavebit.rcwa(backend=backend, psi=psi, n_top=n_top, n_bot=n_bot, theta=theta, phi=phi,
                     fto=fto, wavelength=wavelength, period=period, ucell=ucell,
                     thickness=thickness, device=device, type_complex=dtype)

t0 = time.time()
solver.conv_solve()
print(f'time for efficiency, 1st: ', time.time() - t0)

t0 = time.time()
solver.conv_solve()
print(f'time for efficiency, 2nd: ', time.time() - t0)

t0 = time.time()
solver.calculate_field(res_x=res_x, res_y=res_y, res_z=res_z)
print(f'time for field, 1st: ', time.time() - t0)

t0 = time.time()
solver.calculate_field(res_x=res_x, res_y=res_y, res_z=res_z)
print(f'time for field, 2nd: ', time.time() - t0)

t0 = time.time()
solver.conv_solve_field(res_x=res_x, res_y=res_y, res_z=res_z)
print(f'time for efficiency and field in one step, 1st: ', time.time() - t0)

t0 = time.time()
solver.conv_solve_field(res_x=res_x, res_y=res_y, res_z=res_z)
print(f'time for efficiency and field in one step, 2nd: ', time.time() - t0)


time for efficiency, 1st:  16.540183067321777


time for efficiency, 2nd:  16.759525060653687


time for field, 1st:  1.3695018291473389


time for field, 2nd:  1.2989578247070312


time for efficiency and field in one step, 1st:  18.015563011169434


time for efficiency and field in one step, 2nd:  18.097578048706055


### GPU, 64 bit

In [6]:
if not GPU_AVAILABLE:
    print("SKIPPED: no CUDA GPU on this machine.\n"
          "Nothing was computed — see the note at the top of the notebook.\n"
          "Run this notebook on a CUDA machine to fill in these timings.")
else:
    backend = 2  # PyTorch backend
    device = 1 # GPU;
    dtype = 0
    solver = wavebit.rcwa(backend=backend, psi=psi, n_top=n_top, n_bot=n_bot, theta=theta, phi=phi,
                         fto=fto, wavelength=wavelength, period=period, ucell=ucell,
                         thickness=thickness, device=device, type_complex=dtype)

    t0 = time.time()
    solver.conv_solve()
    print(f'time for efficiency, 1st: ', time.time() - t0)

    t0 = time.time()
    solver.conv_solve()
    print(f'time for efficiency, 2nd: ', time.time() - t0)

    t0 = time.time()
    solver.calculate_field(res_x=res_x, res_y=res_y, res_z=res_z)
    print(f'time for field, 1st: ', time.time() - t0)

    t0 = time.time()
    solver.calculate_field(res_x=res_x, res_y=res_y, res_z=res_z)
    print(f'time for field, 2nd: ', time.time() - t0)

    t0 = time.time()
    solver.conv_solve_field(res_x=res_x, res_y=res_y, res_z=res_z)
    print(f'time for efficiency and field in one step, 1st: ', time.time() - t0)

    t0 = time.time()
    solver.conv_solve_field(res_x=res_x, res_y=res_y, res_z=res_z)
    print(f'time for efficiency and field in one step, 2nd: ', time.time() - t0)


SKIPPED: no CUDA GPU on this machine.
Nothing was computed — see the note at the top of the notebook.
Run this notebook on a CUDA machine to fill in these timings.


### CPU, 32 bit

In [7]:
backend = 2  # PyTorch backend
device = 0  # CPU;
dtype = 1  # 32bit
solver = wavebit.rcwa(backend=backend, psi=psi, n_top=n_top, n_bot=n_bot, theta=theta, phi=phi,
                     fto=fto, wavelength=wavelength, period=period, ucell=ucell,
                     thickness=thickness, device=device, type_complex=dtype)

t0 = time.time()
solver.conv_solve()
print(f'time for efficiency, 1st: ', time.time() - t0)

t0 = time.time()
solver.conv_solve()
print(f'time for efficiency, 2nd: ', time.time() - t0)

t0 = time.time()
solver.calculate_field(res_x=res_x, res_y=res_y, res_z=res_z)
print(f'time for field, 1st: ', time.time() - t0)

t0 = time.time()
solver.calculate_field(res_x=res_x, res_y=res_y, res_z=res_z)
print(f'time for field, 2nd: ', time.time() - t0)

t0 = time.time()
solver.conv_solve_field(res_x=res_x, res_y=res_y, res_z=res_z)
print(f'time for efficiency and field in one step, 1st: ', time.time() - t0)

t0 = time.time()
solver.conv_solve_field(res_x=res_x, res_y=res_y, res_z=res_z)
print(f'time for efficiency and field in one step, 2nd: ', time.time() - t0)


time for efficiency, 1st:  7.7105090618133545


time for efficiency, 2nd:  7.771876811981201


time for field, 1st:  0.6572561264038086


time for field, 2nd:  0.6528232097625732


time for efficiency and field in one step, 1st:  8.38056206703186


time for efficiency and field in one step, 2nd:  8.466909170150757


### GPU, 32 bit

In [8]:
if not GPU_AVAILABLE:
    print("SKIPPED: no CUDA GPU on this machine.\n"
          "Nothing was computed — see the note at the top of the notebook.\n"
          "Run this notebook on a CUDA machine to fill in these timings.")
else:
    backend = 2  # PyTorch backend
    device = 1  # CPU;
    dtype = 1  # 32bit
    solver = wavebit.rcwa(backend=backend, psi=psi, n_top=n_top, n_bot=n_bot, theta=theta, phi=phi,
                         fto=fto, wavelength=wavelength, period=period, ucell=ucell,
                         thickness=thickness, device=device, type_complex=dtype)

    t0 = time.time()
    solver.conv_solve()
    print(f'time for efficiency, 1st: ', time.time() - t0)

    t0 = time.time()
    solver.conv_solve()
    print(f'time for efficiency, 2nd: ', time.time() - t0)

    t0 = time.time()
    solver.calculate_field(res_x=res_x, res_y=res_y, res_z=res_z)
    print(f'time for field, 1st: ', time.time() - t0)

    t0 = time.time()
    solver.calculate_field(res_x=res_x, res_y=res_y, res_z=res_z)
    print(f'time for field, 2nd: ', time.time() - t0)

    t0 = time.time()
    solver.conv_solve_field(res_x=res_x, res_y=res_y, res_z=res_z)
    print(f'time for efficiency and field in one step, 1st: ', time.time() - t0)

    t0 = time.time()
    solver.conv_solve_field(res_x=res_x, res_y=res_y, res_z=res_z)
    print(f'time for efficiency and field in one step, 2nd: ', time.time() - t0)


SKIPPED: no CUDA GPU on this machine.
Nothing was computed — see the note at the top of the notebook.
Run this notebook on a CUDA machine to fill in these timings.
